# 03 — Fine-tuning DistilBERT

We fine-tune `distilbert-base-uncased` on the MDCC text + weak labels. This
notebook is a *walkthrough* of the training pipeline rather than a full
training run — full fine-tuning on CPU takes 15–20 minutes and is invoked
non-interactively by `scripts/04_train_transformer.py`. Here we either:

1. Load the saved trained checkpoint and run inference, **or**
2. Run a tiny 1-epoch fine-tune on a 500-row sample for didactic purposes.

The code below defaults to *(1)* if a saved model is available.

**Educational note.** A transformer reads the whole text with self-attention,
so it can in principle weight context across long spans. However:

- CPU training is slow → we use the smaller DistilBERT variant.
- Long MDCC descriptions (up to 21k chars) must be truncated to `max_length`
  tokens. We use 192 for CPU throughput.
- Weak labels limit the theoretical ceiling — the model can only be as good
  as the labelling rule it's trained against.


In [1]:
# Path-setup boilerplate so the notebook can import src.*
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


project root: /Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)


In [2]:
import json, numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.model_selection import train_test_split

from src import config
from src.evaluation import compute_metrics
from src.models import transformer_classifier as tc


## 1. Decide: load saved or run a mini-training

In [3]:
SAVED = config.MODELS_DIR / 'distilbert'
RUN_MINI = not SAVED.exists()
print('saved checkpoint present:', SAVED.exists())
print('will run mini-training:', RUN_MINI)


saved checkpoint present: True
will run mini-training: False


## 2. Prepare data

In [4]:
df = pd.read_csv(config.PROCESSED_CSV)
df['y'] = df['binary_label'].map(config.LABEL2ID)

if RUN_MINI:
    # Tiny demo run — 500 rows, 1 epoch
    df = df.sample(500, random_state=config.RANDOM_SEED).reset_index(drop=True)

X = df['text'].astype(str).tolist()
y = df['y'].tolist()
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=config.TRAIN_TEST_SPLIT,
    stratify=y, random_state=config.RANDOM_SEED)
print(f'train={len(Xtr):,}  test={len(Xte):,}')


train=11,968  test=2,993


## 3. Train or load

In [5]:
if RUN_MINI:
    Xtr2, Xv, ytr2, yv = train_test_split(
        Xtr, ytr, test_size=0.10, stratify=ytr,
        random_state=config.RANDOM_SEED)
    model, tok, hist = tc.fine_tune(
        Xtr2, ytr2, Xv, yv, epochs=1, save_dir=None)
else:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(str(SAVED))
    model = AutoModelForSequenceClassification.from_pretrained(str(SAVED))
    print('loaded saved DistilBERT from', SAVED)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

loaded saved DistilBERT from /Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)/outputs/models/distilbert


## 4. Evaluate on the test split

In [6]:
ypred, yproba = tc.predict(model, tok, Xte, batch_size=32)
m = compute_metrics(np.array(yte), ypred, yproba, 'DistilBERT',
                    labels=config.LABELS_BINARY)
print(f'accuracy {m.accuracy:.4f}  f1 {m.f1:.4f}  macro_f1 {m.macro_f1:.4f}')
print(m.per_class_report)


accuracy 0.7371  f1 0.6714  macro_f1 0.7261
                  precision    recall  f1-score   support

non_manipulative       0.74      0.82      0.78      1709
    manipulative       0.72      0.63      0.67      1284

        accuracy                           0.74      2993
       macro avg       0.73      0.72      0.73      2993
    weighted avg       0.74      0.74      0.73      2993



## 5. Training history (if produced)

In [7]:
hist_p = config.RESULTS_DIR / 'distilbert_history.json'
if hist_p.exists():
    hist = json.loads(hist_p.read_text())
    pd.DataFrame(hist)
else:
    print('no saved training history; run scripts/04_train_transformer.py')


Expected from the full saved run: train loss 0.624 → 0.468 across two epochs,
val accuracy ~0.77. Test metrics: acc 0.7338, F1 0.6687, macro-F1 0.7231 —
*lower* than the TF-IDF baseline. The full report analyses why (label-lexicon
leakage favours the linear model).
